In [22]:
import pandas as pd
import numpy as np
import sys
import os
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score
import xgboost as xgb

In [23]:
sys.path.append("/home/coralie/workspace/ProjectII/solubility_prediction_nmr/src/controller")
sys.path.append("/home/coralie/workspace/ProjectII/solubility_prediction_nmr/src")

from predictor_runner import generate_ml_ready_file_hansen

In [24]:
solvent_dict = {
    'MeOH': {'Hansen_D': 15.1, 'Hansen_P': 12.3, 'Hansen_H': 22.3},
    'ACN': {'Hansen_D': 15.3, 'Hansen_P': 18.0, 'Hansen_H': 6.1},
    'DMSO': {'Hansen_D': 18.4, 'Hansen_P': 16.4, 'Hansen_H': 10.2},
    'DCM': {'Hansen_D': 18.2, 'Hansen_P': 6.3, 'Hansen_H': 6.1},
    'CHCl3': {'Hansen_D': 17.8, 'Hansen_P': 3.1, 'Hansen_H': 5.7}
}

print(f"Solvants configurés : {list(solvent_dict.keys())}")

Solvants configurés : ['MeOH', 'ACN', 'DMSO', 'DCM', 'CHCl3']


In [25]:
input_path = "~/workspace/ProjectII/solubility_prediction_nmr/data/Fichier final (RT+sol)_with_smiles_Unique_CAS_Unique_CAS.xlsx"
df = pd.read_excel(input_path)
df.columns = df.columns.str.strip()

all_matrix_ids = sorted(df['Matrix ID Name'].dropna().unique().tolist())

print(f"Nombre total de lignes : {len(df)}")
print(f"Nombre de Matrix ID uniques : {len(all_matrix_ids)}")

Nombre total de lignes : 144
Nombre de Matrix ID uniques : 1


In [26]:
def build_all_solvents_df(cleaned_df, solvent_dict, all_matrix_ids):
    all_dfs = []
    for solvent_name in solvent_dict.keys():
        if solvent_name not in cleaned_df.columns:
            continue
        solvent_ml_df = generate_ml_ready_file_hansen(
            cleaned_df, solvent_name, solvent_dict, all_matrix_ids
        )
        solvent_ml_df['Solvent_Name'] = solvent_name
        all_dfs.append(solvent_ml_df)
    return pd.concat(all_dfs, ignore_index=True)

In [27]:
def rank_solvents_for_molecule(model, molwt, rt, matrix_onehot, solvent_dict):
    rows = []
    solvent_names = []
    for solvent_name, hansen in solvent_dict.items():
        row = np.concatenate((
            [molwt, rt],
            matrix_onehot,
            [hansen['Hansen_D'], hansen['Hansen_P'], hansen['Hansen_H']]
        ))
        rows.append(row)
        solvent_names.append(solvent_name)

    X_candidates = np.vstack(rows)
    probas = model.predict_proba(X_candidates)[:, 1]

    ranking = pd.DataFrame({'Solvent': solvent_names, 'Proba_Soluble': probas})
    ranking = ranking.sort_values('Proba_Soluble', ascending=False).reset_index(drop=True)
    return ranking


n_splits = 5
gkf = GroupKFold(n_splits=n_splits)

results_global = []

for fold_num, (train_idx, test_idx) in enumerate(gkf.split(df, groups=df['solute_smiles']), start=1):
    print(f"\n=== Fold {fold_num}/{n_splits} ===")

    df_train = df.iloc[train_idx].reset_index(drop=True)
    df_test = df.iloc[test_idx].reset_index(drop=True)

    train_ml_df = build_all_solvents_df(df_train, solvent_dict, all_matrix_ids)
    test_ml_df = build_all_solvents_df(df_test, solvent_dict, all_matrix_ids)

    X_train = np.hstack((
        train_ml_df[['MolWt', 'RT']].values,
        np.vstack(train_ml_df['Matrix_Onehot'].values),
        np.vstack(train_ml_df['Solvent_Hansen'].values)
    ))
    y_train = train_ml_df['Soluble'].values

    X_test = np.hstack((
        test_ml_df[['MolWt', 'RT']].values,
        np.vstack(test_ml_df['Matrix_Onehot'].values),
        np.vstack(test_ml_df['Solvent_Hansen'].values)
    ))
    y_test = test_ml_df['Soluble'].values

    neg_count = np.sum(y_train == 0)
    pos_count = np.sum(y_train == 1)
    scale_weight = neg_count / pos_count if pos_count > 0 else 1.0

    model = xgb.XGBClassifier(
        n_estimators=100, learning_rate=0.1, max_depth=6,
        random_state=42, eval_metric='logloss',
        scale_pos_weight=scale_weight, n_jobs=-1
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)

    # --- NOUVEAU : évaluation du ranking (top-1 accuracy) ---
    top1_correct = 0
    top1_total = 0

    unique_smiles = test_ml_df['SMILES'].unique()
    for smiles in unique_smiles:
        molecule_rows = test_ml_df[test_ml_df['SMILES'] == smiles]
        first_row = molecule_rows.iloc[0]

        ranking = rank_solvents_for_molecule(
            model, first_row['MolWt'], first_row['RT'], first_row['Matrix_Onehot'], solvent_dict
        )

        top_solvent = ranking.iloc[0]['Solvent']
        actual_row = molecule_rows[molecule_rows['Solvent_Name'] == top_solvent]
        if len(actual_row) > 0 and actual_row.iloc[0]['Soluble'] == 1:
            top1_correct += 1
        top1_total += 1

    top1_accuracy = top1_correct / top1_total if top1_total > 0 else 0
    # --- FIN NOUVEAU ---

    print(f"Accuracy : {acc:.4f} | Precision : {prec:.4f} | Recall : {rec:.4f} | Top-1 (ranking) : {top1_accuracy:.4f}")

    results_global.append({
        'fold': fold_num,
        'N_test': len(y_test),
        'N_molecules': top1_total,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'Top1_Accuracy': top1_accuracy
    })


=== Fold 1/5 ===
Accuracy : 0.6345 | Precision : 0.7816 | Recall : 0.6667 | Top-1 (ranking) : 0.8276

=== Fold 2/5 ===
Accuracy : 0.6828 | Precision : 0.8025 | Recall : 0.6842 | Top-1 (ranking) : 0.8621

=== Fold 3/5 ===
Accuracy : 0.6690 | Precision : 0.7255 | Recall : 0.7872 | Top-1 (ranking) : 0.7586

=== Fold 4/5 ===
Accuracy : 0.6000 | Precision : 0.8028 | Recall : 0.5644 | Top-1 (ranking) : 0.8276

=== Fold 5/5 ===
Accuracy : 0.5714 | Precision : 0.6154 | Recall : 0.6914 | Top-1 (ranking) : 0.5714


In [28]:
results_df = pd.DataFrame(results_global)
print(results_df)

print("\n=== Résumé ===")
print(f"Accuracy       : moyenne = {results_df['Accuracy'].mean():.4f}, écart-type = {results_df['Accuracy'].std():.4f}")
print(f"Precision      : moyenne = {results_df['Precision'].mean():.4f}, écart-type = {results_df['Precision'].std():.4f}")
print(f"Recall         : moyenne = {results_df['Recall'].mean():.4f}, écart-type = {results_df['Recall'].std():.4f}")
print(f"Top-1 Accuracy : moyenne = {results_df['Top1_Accuracy'].mean():.4f}, écart-type = {results_df['Top1_Accuracy'].std():.4f}")

   fold  N_test  N_molecules  Accuracy  Precision    Recall  Top1_Accuracy
0     1     145           29  0.634483   0.781609  0.666667       0.827586
1     2     145           29  0.682759   0.802469  0.684211       0.862069
2     3     145           29  0.668966   0.725490  0.787234       0.758621
3     4     145           29  0.600000   0.802817  0.564356       0.827586
4     5     140           28  0.571429   0.615385  0.691358       0.571429

=== Résumé ===
Accuracy       : moyenne = 0.6315, écart-type = 0.0465
Precision      : moyenne = 0.7456, écart-type = 0.0793
Recall         : moyenne = 0.6788, écart-type = 0.0794
Top-1 Accuracy : moyenne = 0.7695, écart-type = 0.1169
